####Paso 1 - Leer el archivo JSON usando "DataFrameReader de Spark"

In [0]:
%run "../includes/configurations"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
production_country_schema = StructType(fields=[
  StructField("movieId", IntegerType(), True),
  StructField("countryId", IntegerType(), True)
])

In [0]:
production_country_df = spark.read \
  .schema(production_country_schema) \
  .option("multiline", True) \
  .json(f"{bronze_folder_path}/{v_file_date}/production_country")

In [0]:
display(production_country_df)

####Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
production_country_final_df = production_country_df \
    .withColumnsRenamed({"movieId": "movie_id",
                        "countryId": "country_id"}) \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("enviroment", lit("Produccion")) \
    .withColumn("file_date", lit(v_file_date))

In [0]:
display(production_country_final_df)

####Paso 3 - Escribir la salida en formato "Parquet"

In [0]:
#overwrite_partition("movie_silver", "production_countries", "file_date", v_file_date)

In [0]:
#production_country_final_df.write.mode("overwrite").parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/production_country")
#production_country_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.production_countries")
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.file_date = src.file_date'
merge_delta_lake (production_country_final_df, "movie_silver", "production_countries", merge_condition, "file_date")

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.production_countries
GROUP BY file_date;

In [0]:
%sql
SELECT *
FROM movie_silver.production_countries;

In [0]:
#%fs
#ls abfss://silver@moviehistorybymg.dfs.core.windows.net/production_country

In [0]:
#display(spark.read.parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/production_country"))